# InternLM QLoRA 4-bit 指令微調實戰（2026 版）

本 notebook 示範如何在消費級 GPU（例如 RTX 3090 24GB）上，以 4-bit QLoRA 訓練一個大語言模型，並以 SFTTrainer 完成完整指令微調流程。

## 學習目標

1. 理解 `BitsAndBytesConfig` 的正確設定方式，以及 `nf4`、`double_quant`、`bf16 compute_dtype` 的意義。
2. 理解 4-bit / 8-bit / 16-bit 各精度的使用時機。
3. 理解為何 PEFT 前必須先呼叫 `prepare_model_for_kbit_training()`。
4. 以 `tokenizer.apply_chat_template()` 取代硬寫的 `Human:/Assistant:` 模板，確保訓練/推論一致。
5. 以 `trl.SFTTrainer` 取代手刻 `-100` label 遮罩，理解底層 response-only 遮罩原理。
6. 保留底層手刻版作對照，明白 SFTTrainer 替我們自動完成了哪些事。

## 前置知識

- `03-fine-tuning/` 模組：基礎 LoRA 概念與 PEFT 用法
- `04-4bits_training/04-3_4bits_quantization.ipynb`：4-bit 量化原理

## 銜接下一步

- `05-multimodal/` 模組：`apply_chat_template` 同一套機制可攜帶 image/audio token，QLoRA 技術也直接沿用

> **VRAM 需求（InternLM2-7B，4-bit）：** 約 6–8 GB。若 VRAM 不足，可改用 `internlm/internlm2_5-1_8b-chat` 節省約 60% VRAM。

In [ ]:
# ============================================================
# 版本鎖定 — 請在訓練環境執行一次確保相容
# ============================================================
# pip install -q \
#   "transformers>=4.46" \
#   "datasets>=3.0" \
#   "trl>=0.12" \
#   "peft>=0.13" \
#   "accelerate>=1.0" \
#   "bitsandbytes>=0.44" \
#   "evaluate>=0.4" \
#   "safetensors>=0.4" \
#   "torch>=2.4"

import transformers, datasets, trl, peft, accelerate, bitsandbytes, safetensors

print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
print("trl         ", trl.__version__)
print("peft        ", peft.__version__)
print("accelerate  ", accelerate.__version__)
print("bitsandbytes", bitsandbytes.__version__)
print("safetensors ", safetensors.__version__)

In [ ]:
import torch
from transformers import set_seed

# Reproducibility — always set before loading model or data
set_seed(42)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## Step 1 — 匯入套件

2026 版重點套件：
- `BitsAndBytesConfig`：集中管理量化參數，將所有 4-bit/8-bit 設定封裝在單一物件，再透過 `quantization_config` 傳入 `from_pretrained`
- `prepare_model_for_kbit_training`：在掛載 LoRA 之前必須呼叫，讓 frozen 的量化層可正常回傳梯度
- `trl.SFTTrainer` / `SFTConfig`：封裝指令微調的 response-only label 遮罩，取代手刻 `-100` 標籤

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM
import torch

## Step 2 — 載入資料集

使用 HF Hub dataset ID 載入，支援以環境變數 `HF_DATASETS_CACHE` 指定本機快取目錄，無需綁定本機路徑。

**資料集說明**：`silk-road/alpaca-data-gpt4-chinese` 包含約 52K 筆中文指令資料，欄位為 `instruction`、`input`、`output`，與原始 alpaca_data_zh 格式相同。

In [ ]:
# Load from HF Hub — no local path required
# Set HF_DATASETS_CACHE env var if you need a custom cache directory
ds = load_dataset("silk-road/alpaca-data-gpt4-chinese", split="train")
print(ds)
print(ds[0])

## Step 3 — 載入 Tokenizer

2026 版移除硬路徑，改用 HF Hub model ID。

**模型選擇**：
- `internlm/internlm2_5-7b-chat`：約 6–8 GB VRAM（4-bit），效果佳，推薦
- `internlm/internlm2_5-1_8b-chat`：約 2–3 GB VRAM（4-bit），輕量替代

`padding_side='right'` 是 decoder-only 語言模型訓練的關鍵設定：decoder 是從左到右自迴歸生成，pad token 必須靠右對齊，否則 batch > 1 時梯度計算可能不收斂（attention mask 會蓋錯位置）。

In [ ]:
# Replace hard path with HF Hub model ID
# Original: "D:/Pretrained_models/Shanghai_AI_Laboratory/internlm-20b/"
MODEL_ID = "internlm/internlm2_5-7b-chat"
# Lightweight alternative (~2 GB VRAM for 4-bit):
# MODEL_ID = "internlm/internlm2_5-1_8b-chat"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=True,
)

# Must set padding_side='right' for decoder-only models
# When batch size > 1, right-padding keeps the real tokens at
# the same positions across samples, so attention masks align correctly.
tokenizer.padding_side = "right"

# Use eos token as pad if pad token is not defined
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(tokenizer)
print("vocab size:", tokenizer.vocab_size)
print("pad_token_id:", tokenizer.pad_token_id)

## Step 4 — 資料預處理：兩種方式

### 方式 A（底層手刻版，保留作對照教學）

手刻 `-100` label 遮罩，把 instruction 部分標記為 `-100`（不計入 loss），只對 response 部分計算 cross-entropy loss。這樣模型只學「如何回答」，不學「如何提問」。

使用 `tokenizer.apply_chat_template()` 渲染 prompt，確保訓練與推論端使用完全相同的格式，避免分佈不一致。

### 方式 B（2026 推薦）

使用 `tokenizer.apply_chat_template()` + `SFTTrainer`：
- `apply_chat_template` 是跨模型可攜的統一抽象，2026 的多模態訊息（image/audio token）也走同一套機制
- `SFTTrainer` 的 `DataCollatorForCompletionOnlyLM` 自動完成 response-only label 遮罩

以下先展示底層手刻版，讓你理解 `-100` 遮罩原理，然後示範 SFTTrainer 的簡潔寫法。

In [ ]:
# ============================================================
# Method A (bottom-up manual version) — for understanding -100 masking
# ============================================================

def build_chat_messages(example: dict) -> list[dict]:
    """Convert alpaca-format example to messages list."""
    user_content = example["instruction"]
    if example.get("input", "").strip():
        user_content = user_content + "\n" + example["input"]
    return [
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": example["output"]},
    ]


def process_func_manual(example: dict, max_length: int = 384) -> dict:
    """Tokenize one example with manual -100 label masking (for teaching only).

    The -100 positions are ignored by PyTorch's CrossEntropyLoss, so only
    the response tokens contribute to the training loss.
    """
    messages = build_chat_messages(example)

    # apply_chat_template with add_generation_prompt=True gives the prompt portion
    prompt_text = tokenizer.apply_chat_template(
        messages[:-1],  # only the user turn
        tokenize=False,
        add_generation_prompt=True,  # appends the assistant header
    )
    response_text = example["output"]

    prompt_ids  = tokenizer(prompt_text,   add_special_tokens=False)["input_ids"]
    response_ids = tokenizer(response_text, add_special_tokens=False)["input_ids"]

    input_ids      = prompt_ids + response_ids + [tokenizer.eos_token_id]
    attention_mask = [1] * len(input_ids)
    # Mask prompt with -100 so loss is only computed on response tokens
    labels = [-100] * len(prompt_ids) + response_ids + [tokenizer.eos_token_id]

    # Truncate to max_length
    input_ids      = input_ids[:max_length]
    attention_mask = attention_mask[:max_length]
    labels         = labels[:max_length]

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


# Apply to dataset (batched=False here because we split manually; batched=True
# would speed up by ~3-5x for pure tokenization tasks without cross-sample logic)
tokenized_ds_manual = ds.map(
    process_func_manual,
    remove_columns=ds.column_names,
    desc="Tokenizing (manual -100 masking)",
)
print(tokenized_ds_manual)

In [ ]:
# Sanity check: inspect the first sample
print("input_ids[:10]:", tokenized_ds_manual[0]["input_ids"][:10])
print("labels[:10]   :", tokenized_ds_manual[0]["labels"][:10])

# Decode the full input to verify formatting
print("\n--- decoded input ---")
print(tokenizer.decode(tokenized_ds_manual[0]["input_ids"]))

# Decode only the response tokens (filter out -100)
response_ids = [t for t in tokenized_ds_manual[0]["labels"] if t != -100]
print("\n--- decoded response (what the model learns) ---")
print(tokenizer.decode(response_ids))

## Step 5 — 量化設定：BitsAndBytesConfig

### 各精度使用時機

| 精度 | VRAM | 適用情境 |
|------|------|----------|
| fp32 | 基準 x4 | 訓練小模型（< 1B），精度要求極高 |
| bf16 | 基準 x2 | 現代 GPU（Ampere+）主流訓練精度，數值穩定 |
| fp16 | 基準 x2 | 舊 GPU（Volta/Turing），有溢位風險 |
| int8 | 基準 x1 | 純推論，VRAM 減半，精度損失小 |
| nf4 (4-bit) | 基準 x0.5 | VRAM 不足時的 QLoRA 訓練，本 notebook 主題 |

### nf4 vs fp4

- **nf4（Normal Float 4）**：QLoRA 論文（Dettmers et al., 2023）提出，假設權重服從常態分佈，對應 16 個量化點最優。實際效果優於 fp4。
- **fp4（Float Point 4）**：對應 IEEE float 的 4-bit 縮小版，量化點非等間距但非最優分佈。
- **結論**：除非有特殊理由，一律用 `nf4`。

### double_quant 成本效益

`bnb_4bit_use_double_quant=True` 對量化常數再做一次 8-bit 量化，額外壓縮約 0.37 bits/param，7B 模型節省約 200–300 MB VRAM，訓練速度幾乎不受影響。建議開啟。

### 為何 compute_dtype 要用 bf16 而非 fp16

- **bf16** 和 fp32 有相同的指數位元（8 bits），數值範圍相同，不容易溢位或下溢；mantissa 精度（7 bits vs 23 bits）比 fp32 低，但對深度學習已足夠。
- **fp16** 指數只有 5 bits，最大值約 65504，在反向傳播時梯度容易溢位，需要額外的 loss scaling。
- Ampere（RTX 30 系列）以後的 GPU 原生支援 bf16，優先使用。

### device_map='auto' 的語意

`device_map='auto'` 讓 accelerate 自動把模型層分配到可用的 GPU → CPU → disk，單 GPU 放不下時自動 offload，無需手動呼叫 `.cuda()`。

所有量化參數統一封裝在 `BitsAndBytesConfig` 物件，再透過 `quantization_config` 傳入 `from_pretrained`，使設定與模型載入邏輯完全分離。

In [ ]:
# ============================================================
# 4-bit quantization config (2026 standard)
# ============================================================
# VRAM estimate for internlm2_5-7b-chat:
#   fp16  baseline: ~14 GB
#   4-bit nf4     : ~4-5 GB (just weights)
#   + activations + LoRA optimizer states: ~6-8 GB total
# Lightweight: internlm2_5-1_8b-chat reduces to ~2-3 GB

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # Normal Float 4 — best quality
    bnb_4bit_compute_dtype=torch.bfloat16,  # bf16 > fp16: no overflow risk
    bnb_4bit_use_double_quant=True,       # second-level quant saves ~200MB
)
print(bnb_config)

## Step 6 — 載入模型

**關鍵順序**（違反順序會導致訓練 loss 不收斂或報錯）：

1. `from_pretrained` with `quantization_config` → 載入 4-bit 量化模型
2. `prepare_model_for_kbit_training(model)` → 讓 frozen 量化層的 embedding 可以產生梯度
3. `get_peft_model(model, lora_config)` → 插入可訓練的 LoRA adapter

步驟 2 做了三件事：
- 將 embedding layer 轉回 fp32/bf16（量化層不做梯度計算）
- 啟用 gradient checkpointing（節省 activation VRAM，以重算換記憶體）
- 讓 input embeddings 支援 `requires_grad=True`

若跳過步驟 2，訓練時會出現「梯度為 None」或「Input requires grad but was not used」錯誤。

In [ ]:
# Step 6a: Load model with 4-bit quantization
# 2026 unified pattern:
#   device_map='auto'     — let accelerate handle GPU/CPU/disk placement
#   torch_dtype=bfloat16  — numerically stable, no loss scaling needed
#   use_safetensors=True  — safer than pickle, faster to load
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
    trust_remote_code=True,
)
print(model.config)
print("\nModel dtype per layer (sample):")
for name, param in list(model.named_parameters())[:5]:
    print(f"  {name}: shape={param.shape}, dtype={param.dtype}")

In [ ]:
# Step 6b: MUST call before get_peft_model
# This prepares frozen quantized layers to pass gradients through
# and enables gradient checkpointing for activation memory saving.
model = prepare_model_for_kbit_training(model)
print("Model prepared for kbit training")

## Step 7 — LoRA 設定

### target_modules 選擇策略

原始 notebook 只指定 `["q_proj", "k_proj"]`，這是最保守的設定，只微調 attention 的 query 和 key 投影。

2026 版擴展到 `["q_proj", "k_proj", "v_proj", "o_proj"]`（完整 attention），可以顯著提升效果，VRAM 增加有限（LoRA 本身參數極少）。

若希望進一步提升效果，可再加 MLP 層：`"gate_proj", "up_proj", "down_proj"`。

### LoRA 超參解釋

- **r（rank）**：LoRA 分解矩陣的秩，越大參數越多，表達能力越強，但 VRAM 和訓練時間增加
- **lora_alpha**：縮放係數，通常設為 `r` 的 2 倍
- **lora_dropout**：LoRA 層的 dropout，防止過擬合
- **bias='none'**：不微調 bias 項，節省參數

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

# Full attention projection for better quality
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # full attention
    r=8,              # rank
    lora_alpha=16,    # scaling factor (typically 2*r)
    lora_dropout=0.05,
    bias="none",
)
print(lora_config)

# Step 6c: Attach LoRA adapters to the prepared model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected output: ~0.1-0.2% of total parameters are trainable

## Step 8 — SFTTrainer（2026 推薦路徑）

`SFTTrainer` 是 `trl` 提供的 supervised fine-tuning 訓練器，封裝了以下常見邏輯：

1. **response-only label 遮罩**：透過 `DataCollatorForCompletionOnlyLM` 自動找到 response 開始位置，將 prompt 部分標記為 `-100`，無需手寫 `process_func` 中的 `-100` 邏輯
2. **chat template 整合**：`formatting_func` 自動呼叫 `apply_chat_template`
3. **sequence packing**：`SFTConfig(packing=True)` 可把多個短樣本打包成一個 sequence，提高 GPU 利用率
4. **統一介面**：與 `TrainingArguments` / `Trainer` 完全相容，可直接換用

### response_template 的作用

`DataCollatorForCompletionOnlyLM` 需要知道「response 從哪裡開始」，透過 `response_template` 字串找到分界點。此字串必須與 `apply_chat_template` 生成的格式完全一致。

In [ ]:
# ============================================================
# Method B: SFTTrainer (2026 recommended path)
# ============================================================

def formatting_func(example: dict) -> str:
    """Format one example using the model's official chat template.

    Using apply_chat_template guarantees that the prompt format at
    training time matches the format at inference time.
    This is critical — a mismatch causes the model to underperform.
    """
    messages = build_chat_messages(example)
    # tokenize=False returns the formatted string for SFTTrainer to tokenize
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,  # False for training (response is included)
    )


# Verify the formatting
print("Sample formatted input:")
print(formatting_func(ds[0]))
print()

In [ ]:
from trl import DataCollatorForCompletionOnlyLM

# DataCollatorForCompletionOnlyLM automatically masks prompt tokens with -100
# It scans each sample for response_template and masks everything before it.
#
# For InternLM2, the assistant turn starts with <|im_start|>assistant
# Verify this by checking the formatted output above.
response_template = "<|im_start|>assistant"
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer,
)

## Step 9 — 訓練設定（SFTConfig）

### 重要 TrainingArguments 欄位解釋

| 參數 | 值 | 原因 |
|------|----|------|
| `bf16` | `True` | bf16 比 fp16 穩定，不需 loss scaling |
| `warmup_ratio` | `0.1` | 前 10% steps 線性升溫，穩定早期梯度 |
| `lr_scheduler_type` | `'cosine'` | 餘弦衰減比線性衰減效果更好 |
| `max_grad_norm` | `1.0` | 梯度裁剪，防止梯度爆炸 |
| `save_safetensors` | `True` | 儲存安全格式，防止 pickle 漏洞 |
| `optim` | `adamw_torch_fused` | fused 版本在同等 VRAM 下更快 |
| `seed` | `42` | 可重現性 |

### effective batch size

`effective_batch_size = per_device_train_batch_size × gradient_accumulation_steps × num_gpus`

本範例：`1 × 32 × 1 = 32`。梯度累積讓小 VRAM 的 GPU 等效大 batch 訓練。

In [ ]:
from trl import SFTConfig

# SFTConfig extends TrainingArguments with SFT-specific options
# such as max_seq_length, packing, dataset_text_field, etc.
sft_args = SFTConfig(
    output_dir="./internlm-qlora-sft",

    # Batch and gradient settings
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,    # effective batch = 1 * 32 = 32
    gradient_checkpointing=True,       # saves activation memory at cost of recompute

    # Precision
    bf16=True,                         # bf16 > fp16: no overflow risk

    # Optimizer and scheduler
    learning_rate=2e-4,
    optim="adamw_torch_fused",         # fused AdamW: faster than paged_adamw_32bit
    warmup_ratio=0.1,                  # linear warmup for first 10% of steps
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,

    # Training duration
    num_train_epochs=1,

    # Logging and saving
    logging_steps=10,
    save_steps=500,
    save_total_limit=2,
    save_safetensors=True,             # safe serialization, no pickle

    # Reproducibility
    seed=42,

    # SFT-specific: sequence length cap
    max_seq_length=384,
    dataset_text_field=None,           # we use formatting_func instead
    packing=False,                     # set True to pack short samples (faster)
)
print(sft_args)

## Step 10 — 建立 SFTTrainer 並開始訓練

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=ds.select(range(6000)),   # use 6000 samples for demo
    formatting_func=formatting_func,
    data_collator=collator,
    processing_class=tokenizer,
)

# Show trainable parameter count again after SFTTrainer setup
model.print_trainable_parameters()

In [ ]:
# Start training
# Expected output: training loss should decrease from ~2.x to ~1.x over 6000 samples
trainer.train()

## Step 11 — 儲存 LoRA Adapter

LoRA 訓練完成後，只需儲存 adapter 權重（約幾十 MB），不需儲存完整模型（幾 GB）。推論時再動態合併。

`safe_serialization=True` 使用 safetensors 格式，比 pickle 更安全（無 arbitrary code execution 風險）且載入速度更快（支援 memory-mapped 載入）。

In [ ]:
# Save only the LoRA adapter weights (much smaller than full model)
adapter_output_dir = "./internlm-qlora-adapter"
model.save_pretrained(adapter_output_dir, safe_serialization=True)
tokenizer.save_pretrained(adapter_output_dir)
print(f"LoRA adapter saved to: {adapter_output_dir}")

## Step 12 — 推論

### apply_chat_template 在推論端的使用

訓練端與推論端**必須使用完全相同的 chat template**。

2026 版兩端統一走 `apply_chat_template`，換模型時只需改 `MODEL_ID`，其他不用動。這是跨模型可攜的關鍵設計：每個模型的 tokenizer 自帶正確的模板，呼叫者無需知道格式細節。

In [ ]:
model.eval()

def chat(user_input: str, max_new_tokens: int = 256) -> str:
    """Run inference using the same chat template as training."""
    messages = [{"role": "user", "content": user_input}]

    # add_generation_prompt=True appends the assistant header so the model
    # knows it should generate the response (not the user's next message)
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode only the generated tokens (skip the prompt)
    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


# Test inference
response = chat("請列出保持健康的三個建議。")
print("Response:")
print(response)

## Step 13 — 合併 LoRA 權重（選用）

`merge_and_unload()` 把 LoRA 的 $\Delta W = BA$ 合併回原始權重 $W$，生成一個普通的 full-weight 模型，推論時無 LoRA overhead，但 VRAM 回到未量化前的大小。

**一般情境建議**：
- **保留 adapter**（不合併）：部署時動態載入，靈活切換不同 adapter，VRAM 節省
- **合併後儲存**：需要將模型發布到 HF Hub 或給不支援 PEFT 的框架使用時

In [ ]:
# Optional: merge LoRA weights back into the base model
# This creates a standard (non-PEFT) model for deployment
# Note: the merged model will be in bf16, not 4-bit quantized
#       VRAM requirement returns to full model size (~14 GB for 7B)

merged_model = model.merge_and_unload()
print("LoRA merged. Model type:", type(merged_model))

# Save merged model with safe serialization
merged_output_dir = "./internlm-qlora-merged"
merged_model.save_pretrained(merged_output_dir, safe_serialization=True)
tokenizer.save_pretrained(merged_output_dir)
print(f"Merged model saved to: {merged_output_dir}")

## Step 14 — 推送到 HF Hub（選用）

`push_to_hub` 可以把模型或 adapter 推送到 Hugging Face Hub，方便分享與部署。最小 model card 應包含：`id2label`（若有分類任務）、語言標籤、授權、任務標籤。

In [ ]:
# Optional: push LoRA adapter to HF Hub
# Requires: huggingface-cli login (or HF_TOKEN env var)
#
# hub_model_id = "your-username/internlm2_5-7b-chat-qlora-alpaca-zh"
# model.push_to_hub(
#     hub_model_id,
#     safe_serialization=True,
#     commit_message="QLoRA adapter trained on alpaca-data-gpt4-chinese",
# )
# tokenizer.push_to_hub(hub_model_id)
#
# Minimal model card fields to set before pushing:
# model.config.update({
#     "language": ["zh"],
#     "license": "apache-2.0",
#     "library_name": "peft",
# })
print("push_to_hub example shown above (commented out — requires HF token)")

## 小結

本 notebook 完整示範了 2026 版 QLoRA 4-bit 指令微調的標準流程：

| 主題 | 2026 做法 |
|------|-----------|
| 量化設定 | `BitsAndBytesConfig` 封裝所有參數，透過 `quantization_config` 傳入 |
| 精度 | `torch.bfloat16`（bf16），數值穩定無需 loss scaling |
| 模型載入 | HF Hub ID + `device_map='auto'` + `use_safetensors=True` |
| PEFT 前置 | 先 `prepare_model_for_kbit_training`，再 `get_peft_model` |
| Chat template | `tokenizer.apply_chat_template()`，訓練與推論端統一 |
| 指令微調 | `SFTTrainer` + `DataCollatorForCompletionOnlyLM` |
| 儲存格式 | `safe_serialization=True`（safetensors） |

### 練習題

1. **精度對比**：將 `bnb_4bit_compute_dtype` 改為 `torch.float16`，觀察訓練是否需要 loss scaling 以及 loss 曲線是否有異常波動。
2. **量化深度**：改用 `load_in_8bit=True`（BitsAndBytesConfig 設定），對比 4-bit 和 8-bit 在 VRAM 使用量和訓練速度上的差異。
3. **LoRA 目標模組**：在 `target_modules` 中加入 `["gate_proj", "up_proj", "down_proj"]`，觀察 trainable parameters 百分比的變化，以及訓練效果是否提升。
4. **Sequence packing**：將 `SFTConfig` 中 `packing=True` 開啟，對比 steps/sec 與 packing=False 的差異，理解 GPU 利用率的提升。
5. **apply_chat_template 驗證**：手動用 `tokenizer.apply_chat_template([{"role":"user","content":"你好"}], tokenize=False, add_generation_prompt=True)` 列印模板，確認推論時的格式與訓練時完全一致。